# GSM8K Accuracy Benchmark

Zero-shot **GSM8K exact-match** on **Qwen/Qwen3.5-4B** (swap to `Qwen/Qwen3.5-9B` after this pipeline is clean). This notebook only generates answers and scores them. Jacobian-lens / J-space readout comes later.

This is **not** the gold-next-token controllability helpers in `jlens.benchmark`. Those measure steered token rank; this measures whether the numeric answer matches GSM8K gold.

Adjust `SMOKE_TEST` at the top (`True` = 8 items) before a full 1319-item run.


### Optional: install dataset dependency

Run once if `datasets` is missing.


In [2]:
# %pip install datasets


In [3]:
import datetime
import json
import re
from fractions import Fraction
from pathlib import Path

import torch
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm

# --- Model ---
MODEL_NAME = "Qwen/Qwen3.5-4B"
# MODEL_NAME = "Qwen/Qwen3.5-9B"  # swap after the 4B run is clean
MODEL_TAG = MODEL_NAME.rsplit("/", 1)[-1].lower()

# --- Benchmark knobs ---
SMOKE_TEST = True  # False: full GSM8K test split (1319)
ENABLE_THINKING = True  # Qwen3.5 thinks by default; math benefits from it
N_EXAMPLES = 8 if SMOKE_TEST else None  # None = all test items
MAX_NEW_TOKENS = 256 if SMOKE_TEST else 1024
SAVE_EVERY = 1 if SMOKE_TEST else 10
SEED = 0

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_PATH = RESULTS_DIR / f"gsm8k_benchmark_{MODEL_TAG}.json"

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch_dtype = torch.bfloat16
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
    torch_dtype = torch.float16
else:
    device = torch.device("cpu")
    torch_dtype = torch.float32

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch_dtype, trust_remote_code=True
).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"model: {MODEL_NAME} on {device} / {torch_dtype}")
print(f"thinking: {ENABLE_THINKING}  max_new_tokens: {MAX_NEW_TOKENS}  smoke: {SMOKE_TEST}")
print(f"results: {RESULTS_PATH}")


/Users/cyb/Documents/GitHub/jspace-research/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 27869.13it/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:00<00:00, 4592.60it/s]


model: Qwen/Qwen3.5-4B on mps / torch.float16
thinking: True  max_new_tokens: 256  smoke: True
results: results/gsm8k_benchmark_qwen3.5-4b.json


## Prompting, generation, and scoring

Official Qwen math wording, greedy decode, then extract `\boxed{}` (fallback: `####`, then last number). Gold is the GSM8K `####` field.


In [4]:
MATH_INSTRUCTION = (
    "Please reason step by step, and put your final answer within \\boxed{}."
)
_NUMBER_RE = re.compile(
    r"[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?:/\d+)?"
)
_THINK_BLOCK_RE = re.compile(
    r"<(?:redacted_)?think(?:ing)?>.*?</(?:redacted_)?think(?:ing)?>",
    flags=re.DOTALL | re.IGNORECASE,
)


def strip_thinking(text: str) -> str:
    """Drop Qwen thinking blocks; score only the final answer text."""
    text = _THINK_BLOCK_RE.sub("", text)
    for close in ("</think>", "</thinking>", "</redacted_thinking>"):
        if close in text:
            text = text.rsplit(close, 1)[-1]
    return text.strip()


def build_prompt(question: str) -> str:
    user = f"{question.strip()}\n\n{MATH_INSTRUCTION}"
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )


@torch.no_grad()
def generate_completion(prompt: str) -> str:
    enc = tokenizer(prompt, return_tensors="pt").to(hf_model.device)
    out = hf_model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0, enc.input_ids.shape[1] :], skip_special_tokens=False)


def gold_from_gsm8k(answer: str) -> str:
    if "####" not in answer:
        raise ValueError(f"GSM8K answer missing ####: {answer!r}")
    return answer.rsplit("####", 1)[-1].strip()


def _extract_balanced_boxed(text: str) -> str | None:
    key = r"\boxed{"
    start = text.rfind(key)
    if start < 0:
        return None
    i = start + len(key)
    depth = 1
    while i < len(text):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start + len(key) : i].strip()
        i += 1
    return None


def extract_prediction(text: str) -> str | None:
    """Prefer last \\boxed{}; then ####; then the last number in the final text."""
    final = strip_thinking(text)
    boxed = _extract_balanced_boxed(final) or _extract_balanced_boxed(text)
    if boxed:
        inner = boxed.strip()
        inner = re.sub(r"^\\text\{(.*)\}$", r"\1", inner).strip()
        return inner
    if "####" in final:
        return final.rsplit("####", 1)[-1].strip().splitlines()[0].strip()
    matches = list(_NUMBER_RE.finditer(final))
    if matches:
        return matches[-1].group(0)
    return None


def canonicalize_number(raw: str | None) -> str | None:
    if raw is None:
        return None
    text = raw.strip()
    text = text.replace(",", "").replace("$", "").replace("%", "")
    text = text.strip().rstrip(".")
    text = re.sub(r"^\\(?:mathrm|mathbf|text)\{(.*)\}$", r"\1", text).strip()
    frac_m = re.fullmatch(r"\\(?:d|t)?frac\{([^{}]+)\}\{([^{}]+)\}", text)
    if frac_m:
        text = f"{frac_m.group(1)}/{frac_m.group(2)}"
    if not text:
        return None
    if "/" in text and text.count("/") == 1:
        num, den = text.split("/")
        try:
            value = float(Fraction(num.strip()) / Fraction(den.strip()))
        except (ValueError, ZeroDivisionError):
            return None
    else:
        try:
            value = float(text)
        except ValueError:
            matches = list(_NUMBER_RE.finditer(text))
            if not matches:
                return None
            try:
                value = float(matches[-1].group(0).replace(",", ""))
            except ValueError:
                return None
    if abs(value - round(value)) < 1e-9:
        return str(int(round(value)))
    return format(value, ".10g")


def answers_match(pred_raw: str | None, gold_raw: str) -> bool:
    pred = canonicalize_number(pred_raw)
    gold = canonicalize_number(gold_raw)
    return pred is not None and gold is not None and pred == gold


print("helpers loaded")


helpers loaded


## Run GSM8K

Loads `openai/gsm8k` test. Incremental JSON writes so an interrupted MPS run can resume.


In [5]:
def summarize(records: list[dict]) -> dict:
    n = len(records)
    n_correct = sum(1 for row in records if row["correct"])
    n_parse_fail = sum(1 for row in records if row["pred"] is None)
    return {
        "n": n,
        "n_correct": n_correct,
        "accuracy": (n_correct / n) if n else 0.0,
        "n_parse_fail": n_parse_fail,
    }


def build_payload(records: list[dict]) -> dict:
    return {
        "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "config": {
            "model_name": MODEL_NAME,
            "model_tag": MODEL_TAG,
            "smoke_test": SMOKE_TEST,
            "enable_thinking": ENABLE_THINKING,
            "max_new_tokens": MAX_NEW_TOKENS,
            "n_requested": N_EXAMPLES,
            "seed": SEED,
        },
        "summary": summarize(records),
        "records": records,
    }


def save_results(records: list[dict]) -> None:
    RESULTS_PATH.write_text(json.dumps(build_payload(records), indent=2), encoding="utf-8")


ds = load_dataset("openai/gsm8k", "main", split="test")
if N_EXAMPLES is not None:
    ds = ds.select(range(min(N_EXAMPLES, len(ds))))

records: list[dict] = []
done_indices: set[int] = set()
if RESULTS_PATH.exists():
    previous = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
    prev_cfg = previous.get("config", {})
    same_run = (
        prev_cfg.get("model_name") == MODEL_NAME
        and prev_cfg.get("enable_thinking") == ENABLE_THINKING
        and prev_cfg.get("max_new_tokens") == MAX_NEW_TOKENS
        and prev_cfg.get("smoke_test") == SMOKE_TEST
        and prev_cfg.get("n_requested") == N_EXAMPLES
    )
    if same_run:
        records = previous.get("records", [])
        done_indices = {int(row["index"]) for row in records}
        print(f"resuming {RESULTS_PATH} with {len(records)} existing records")
    else:
        print(f"ignoring {RESULTS_PATH}: config mismatch, starting a fresh run")

n_total = len(ds)
print(f"evaluating {n_total} GSM8K test items")

for index, row in enumerate(tqdm(ds, total=n_total)):
    if index in done_indices:
        continue
    question = str(row["question"])
    gold = gold_from_gsm8k(str(row["answer"]))
    prompt = build_prompt(question)
    generation = generate_completion(prompt)
    pred_raw = extract_prediction(generation)
    pred = canonicalize_number(pred_raw)
    correct = answers_match(pred_raw, gold)
    records.append(
        {
            "index": index,
            "question": question,
            "gold": canonicalize_number(gold),
            "gold_raw": gold,
            "pred": pred,
            "pred_raw": pred_raw,
            "correct": correct,
            "generation": generation,
            "final_text": strip_thinking(generation),
        }
    )
    if len(records) % SAVE_EVERY == 0 or index + 1 == n_total:
        save_results(records)

save_results(records)
summary = summarize(records)
print(
    f"accuracy: {summary['n_correct']}/{summary['n']}"
    f" = {summary['accuracy']:.3f}  parse_fail: {summary['n_parse_fail']}"
)
print(f"wrote {RESULTS_PATH}")


Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 499078.66 examples/s]


evaluating 8 GSM8K test items


100%|██████████| 8/8 [03:05<00:00, 23.18s/it]

accuracy: 0/8 = 0.000  parse_fail: 0
wrote results/gsm8k_benchmark_qwen3.5-4b.json


## Summary


In [6]:
summary = summarize(records)
print(f"model:     {MODEL_NAME}")
print(f"n:         {summary['n']}")
print(f"correct:   {summary['n_correct']}")
print(f"accuracy:  {summary['accuracy']:.3%}")
print(f"parse fail:{summary['n_parse_fail']}")

wrong = [row for row in records if not row["correct"]]
print(f"\nwrong: {len(wrong)}")
for row in wrong[:5]:
    print("-" * 60)
    print(f"[{row['index']}] gold={row['gold']}  pred={row['pred']}")
    q = row["question"].replace("\n", " ")
    print(f"Q: {q[:240]}{'…' if len(q) > 240 else ''}")
    preview = row["final_text"].replace("\n", " ")
    print(f"A: {preview[:320]}{'…' if len(preview) > 320 else ''}")


model:     Qwen/Qwen3.5-4B
n:         8
correct:   0
accuracy:  0.000%
parse fail:0

wrong: 8
------------------------------------------------------------
[0] gold=18  pred=3
Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does sh…
A: Here's my thought process for solving this word problem:  1.  **Analyze the Request:**     *   **Subject:** Janet's ducks.     *   **Production:** 16 eggs per day.     *   **Consumption (Breakfast):** 3 eggs.     *   **Consumption (Baking):** 4 eggs.     *   **Sales:** Remainder sold at the farmers' market.     *   **P…
------------------------------------------------------------
[1] gold=3  pred=1
Q: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
A: Here's my thought process for solving this word problem:  1.  *

## Next: J-lens / J-space (not in this notebook)

Once this accuracy baseline is stable, add observation on the same prompts:

1. Wrap the loaded HF model with `jlens.from_hf(hf_model, tokenizer)`.
2. Load the matching Neuronpedia Jacobian lens (`qwen3.5-4b` file for 4B). The 9B lens in this repo is fit on **`Qwen/Qwen3.5-9B-Base`**, not instruct 9B.
3. Score J-space concepts / ranks on the GSM8K prompts (and optionally on generated traces) without changing the generation protocol above.

Keep this notebook as the behavioral score; layer a readout pass on top rather than mixing steering into the accuracy loop.
